# Orientador - Archivos de texto y archivos CSV

## 1. Concepto y organización de un archivo

**a) ¿Qué significa que la información persista en el tiempo?**
Que los datos siguen existiendo después de que el programa termina. Al guardarse en un dispositivo no volátil (disco, SSD), se pueden recuperar en ejecuciones posteriores, incluso después de apagar la computadora.

**b) ¿Qué diferencias encuentran entre una variable en memoria y un archivo?**
- La variable vive en la memoria RAM y se pierde cuando el programa termina. El archivo se guarda en disco y se conserva.
- El acceso a una variable es muy rápido. El acceso a un archivo es más lento porque implica operaciones de entrada/salida.
- La variable es exclusiva del programa que la creó. El archivo puede ser leído por otros programas o por el mismo programa en otra ejecución.
- Trabajar con archivos puede fallar (archivo inexistente, sin permisos, disco lleno), por eso requiere manejo de errores.

**c) ¿Qué ventajas y limitaciones presenta almacenar los datos como texto?**
Ventajas: se puede abrir y leer con cualquier editor, es fácil de revisar y corregir a mano, y es compatible entre distintos sistemas y lenguajes.
Limitaciones: todo se guarda como caracteres, así que los números hay que convertirlos al leer (y eso puede fallar). Ocupa más espacio que un formato binario, y la estructura (delimitadores, cantidad de campos) depende de una convención que el programa debe respetar y validar.

## 2. Apertura, procesamiento y cierre

Modos de apertura:
- **"rt"** (lectura): el archivo debe existir; si no existe se produce `FileNotFoundError`.
- **"wt"** (escritura): crea el archivo; si ya existe, borra todo su contenido y empieza de cero.
- **"at"** (agregado): crea el archivo si no existe; si existe, escribe al final sin borrar lo anterior.

**Función de cada elemento:**
- `open()` abre el archivo y devuelve un objeto archivo. Recibe el nombre, el modo y la codificación (`utf-8` para que se guarden bien las tildes y la ñ).
- `write()` escribe una cadena en el archivo. Solo recibe strings y no agrega el salto de línea solo, por eso hay que poner `\n` al final de cada registro.
- `with` se encarga de cerrar el archivo automáticamente al terminar el bloque, incluso si ocurre una excepción adentro. Así no hace falta llamar a `close()`.

In [ ]:
with open("notas.txt", "wt", encoding="utf-8") as archivo:
    archivo.write("Primer registro\n")
    archivo.write("Segundo registro\n")
    archivo.write("Tercer registro\n")

# Verificación: leemos lo que quedó grabado
with open("notas.txt", "rt", encoding="utf-8") as archivo:
    for linea in archivo:
        print(linea.rstrip("\n"))

## 3. Escritura de registros de texto

La función `crear_alumnos()` pide legajo, nombre y apellido, y comisión hasta que se ingrese el legajo -1 (que no se graba). Cada registro se guarda en una línea con el formato `legajo;nombre;comision`.

Validaciones antes de escribir:
- El legajo debe ser positivo.
- El nombre no puede estar vacío (ni contener `;`, porque rompería el formato del registro).
- La comisión debe ser un número entre 1 y 10.

La función recibe el modo como parámetro para poder probar "wt" y "at".

In [ ]:
def pedir_entero(mensaje):
    """Pide un número entero hasta que el usuario ingrese uno válido."""
    while True:
        try:
            return int(input(mensaje))
        except ValueError:
            print("Ingresá un número entero válido.")


def crear_alumnos(modo="wt"):
    try:
        with open("alumnos.txt", modo, encoding="utf-8") as archivo:
            legajo = pedir_entero("Legajo (-1 para terminar): ")
            while legajo != -1:
                if legajo <= 0:
                    print("El legajo debe ser positivo. No se grabó.")
                else:
                    nombre = input("Nombre y apellido: ").strip()
                    if nombre == "" or ";" in nombre:
                        print("Nombre inválido (vacío o con ';'). No se grabó.")
                    else:
                        comision = pedir_entero("Comisión (1 a 10): ")
                        if comision < 1 or comision > 10:
                            print("La comisión debe estar entre 1 y 10. No se grabó.")
                        else:
                            archivo.write(f"{legajo};{nombre};{comision}\n")
                            print("Alumno grabado.")
                legajo = pedir_entero("Legajo (-1 para terminar): ")
    except OSError as error:
        print("No se pudo crear o escribir el archivo:", error)

In [ ]:
# Primera prueba con modo "wt"
crear_alumnos("wt")

**Prueba con "wt" por segunda vez:** al ejecutar de nuevo la función con modo "wt", los alumnos cargados en la primera ejecución se perdieron. El modo "wt" vacía el archivo al abrirlo, así que solo quedan los registros de la última carga.

**Prueba con "at":** con modo "at" los registros nuevos se agregan al final y los anteriores se conservan. El archivo va acumulando alumnos en cada ejecución.

Conclusión: "wt" sirve para crear el archivo desde cero; "at" sirve para seguir agregando registros a un archivo existente.

In [ ]:
# Prueba con modo "at": se agregan alumnos sin borrar los anteriores
crear_alumnos("at")

## 4. Lectura y procesamiento secuencial

Para leer registro por registro se recorre el archivo con `for linea in archivo`, que trae una línea por vez sin cargar todo el archivo en memoria.

La función `leer_alumno()` centraliza la validación de cada línea: quita solo el salto de línea con `rstrip("\n")`, separa por `;`, verifica que haya exactamente 3 campos y recién ahí convierte el legajo y la comisión. Si algo falla devuelve `None`.

Se usa `rstrip("\n")` y no `strip()` porque `strip()` también borraría espacios al principio y al final, que podrían ser parte del dato.

El punto d) (líneas que no respetan el formato) se informa dentro de `mostrar_alumnos()`, indicando el número de línea.

In [ ]:
def leer_alumno(linea):
    """Devuelve (legajo, nombre, comision) o None si la línea no respeta el formato."""
    campos = linea.rstrip("\n").split(";")
    if len(campos) != 3:
        return None
    try:
        legajo = int(campos[0])
        comision = int(campos[2])
    except ValueError:
        return None
    nombre = campos[1]
    if legajo <= 0 or nombre.strip() == "":
        return None
    return legajo, nombre, comision


# a) y d) Mostrar registros válidos e informar los inválidos
def mostrar_alumnos():
    try:
        with open("alumnos.txt", "rt", encoding="utf-8") as archivo:
            nro_linea = 0
            invalidas = 0
            for linea in archivo:
                nro_linea += 1
                alumno = leer_alumno(linea)
                if alumno is None:
                    invalidas += 1
                    print(f"Línea {nro_linea} con formato inválido: {linea.rstrip()}")
                else:
                    legajo, nombre, comision = alumno
                    print(f"Legajo: {legajo:<8} Nombre: {nombre:<25} Comisión: {comision}")
            print(f"\nLíneas inválidas encontradas: {invalidas}")
    except FileNotFoundError:
        print("El archivo alumnos.txt no existe. Primero ejecutá crear_alumnos().")
    except OSError as error:
        print("No se pudo leer el archivo:", error)


# b) Alumnos con legajo mayor a un límite
def alumnos_legajo_mayor(limite=10000):
    try:
        with open("alumnos.txt", "rt", encoding="utf-8") as archivo:
            encontrados = 0
            for linea in archivo:
                alumno = leer_alumno(linea)
                if alumno is not None and alumno[0] > limite:
                    legajo, nombre, comision = alumno
                    print(f"Legajo: {legajo:<8} Nombre: {nombre:<25} Comisión: {comision}")
                    encontrados += 1
            if encontrados == 0:
                print(f"No hay alumnos con legajo mayor a {limite}.")
    except FileNotFoundError:
        print("El archivo alumnos.txt no existe.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)


# c) Cantidad de alumnos por comisión usando un diccionario
def contar_por_comision():
    conteo = {}
    try:
        with open("alumnos.txt", "rt", encoding="utf-8") as archivo:
            for linea in archivo:
                alumno = leer_alumno(linea)
                if alumno is not None:
                    comision = alumno[2]
                    conteo[comision] = conteo.get(comision, 0) + 1
    except FileNotFoundError:
        print("El archivo alumnos.txt no existe.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)
    return conteo

In [ ]:
print("=== a) y d) Todos los alumnos ===")
mostrar_alumnos()

print("\n=== b) Legajos mayores a 10000 ===")
alumnos_legajo_mayor(10000)

print("\n=== c) Alumnos por comisión ===")
conteo = contar_por_comision()
for comision in sorted(conteo):
    print(f"Comisión {comision}: {conteo[comision]} alumno(s)")

## 5. Archivos CSV y encabezados

Se crea `productos.csv` con el encabezado `codigo;descripcion;cantidad;precio` y delimitador `;`.

Funciones:
- `crear_productos()`: crea el archivo (modo "wt") y escribe solo el encabezado.
- `cargar_productos()`: agrega productos (modo "at") hasta ingresar el código FIN, que no se guarda.
- `leer_producto()`: valida una línea (4 campos, cantidad entera >= 0, precio > 0).
- `procesar_productos()`: lee el archivo una sola vez, muestra el importe de cada producto (cantidad × precio), el importe total y los productos con cantidad menor a 5.

El encabezado se omite de forma controlada: se lee con `readline()` antes del `for` y se verifica que sea el esperado, así nunca se procesa como si fuera un producto.

In [ ]:
ARCHIVO_PRODUCTOS = "productos.csv"
ENCABEZADO_PRODUCTOS = "codigo;descripcion;cantidad;precio"


def pedir_real(mensaje):
    """Pide un número real hasta que el usuario ingrese uno válido."""
    while True:
        try:
            return float(input(mensaje))
        except ValueError:
            print("Ingresá un número válido (usá punto para los decimales).")


def crear_productos():
    try:
        with open(ARCHIVO_PRODUCTOS, "wt", encoding="utf-8") as archivo:
            archivo.write(ENCABEZADO_PRODUCTOS + "\n")
        print("Archivo productos.csv creado con su encabezado.")
    except OSError as error:
        print("No se pudo crear el archivo:", error)


def cargar_productos():
    try:
        with open(ARCHIVO_PRODUCTOS, "at", encoding="utf-8") as archivo:
            codigo = input("Código (FIN para terminar): ").strip()
            while codigo.upper() != "FIN":
                if codigo == "" or ";" in codigo:
                    print("Código inválido. No se grabó.")
                else:
                    descripcion = input("Descripción: ").strip()
                    cantidad = pedir_entero("Cantidad: ")
                    precio = pedir_real("Precio: ")
                    if descripcion == "" or ";" in descripcion:
                        print("Descripción inválida. No se grabó.")
                    elif cantidad < 0:
                        print("La cantidad no puede ser negativa. No se grabó.")
                    elif precio <= 0:
                        print("El precio debe ser mayor que cero. No se grabó.")
                    else:
                        archivo.write(f"{codigo};{descripcion};{cantidad};{precio}\n")
                        print("Producto grabado.")
                codigo = input("Código (FIN para terminar): ").strip()
    except OSError as error:
        print("No se pudo escribir el archivo:", error)


def leer_producto(linea):
    """Devuelve (codigo, descripcion, cantidad, precio) o None si la línea es inválida."""
    campos = linea.rstrip("\n").split(";")
    if len(campos) != 4:
        return None
    try:
        cantidad = int(campos[2])
        precio = float(campos[3])
    except ValueError:
        return None
    codigo = campos[0].strip()
    descripcion = campos[1].strip()
    if codigo == "" or descripcion == "" or cantidad < 0 or precio <= 0:
        return None
    return codigo, descripcion, cantidad, precio


def procesar_productos():
    try:
        with open(ARCHIVO_PRODUCTOS, "rt", encoding="utf-8") as archivo:
            encabezado = archivo.readline().rstrip("\n")
            if encabezado != ENCABEZADO_PRODUCTOS:
                print("El archivo no tiene el encabezado esperado. No se procesa.")
                return

            total = 0
            stock_bajo = []
            nro_linea = 1
            print(f"{'Código':<8} {'Descripción':<20} {'Cant.':>6} {'Precio':>12} {'Importe':>14}")
            for linea in archivo:
                nro_linea += 1
                producto = leer_producto(linea)
                if producto is None:
                    print(f"Línea {nro_linea} inválida, se omite: {linea.rstrip()}")
                else:
                    codigo, descripcion, cantidad, precio = producto
                    importe = cantidad * precio
                    total += importe
                    print(f"{codigo:<8} {descripcion:<20} {cantidad:>6} {precio:>12.2f} {importe:>14.2f}")
                    if cantidad < 5:
                        stock_bajo.append(f"{codigo} - {descripcion} ({cantidad} unidades)")

            print(f"\nImporte total: ${total:.2f}")
            print("\nProductos con cantidad menor a 5:")
            if len(stock_bajo) == 0:
                print("Ninguno.")
            for producto in stock_bajo:
                print("-", producto)
    except FileNotFoundError:
        print("El archivo productos.csv no existe. Primero ejecutá crear_productos().")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

In [ ]:
crear_productos()
cargar_productos()
procesar_productos()

## 6. Manejo de excepciones

En los ejercicios anteriores no se usó `except:` solo, sino excepciones específicas:
- **a) OSError:** en todas las funciones que abren, crean o escriben archivos. Para lectura se captura antes `FileNotFoundError` (que es un tipo de OSError) para dar un mensaje más claro.
- **b) ValueError:** al convertir legajos, comisiones, cantidades, precios y porcentajes, tanto al pedir datos por teclado (`pedir_entero`, `pedir_real`) como al leer registros del archivo.
- **c) Registros incompletos:** se controla con `len(campos)` antes de convertir. Las funciones `leer_...()` devuelven `None` y el registro se informa y se omite sin cortar el procesamiento.

**try-except-finally vs with:**
Con `finally` el cierre del archivo se hace a mano. La variable del archivo debe inicializarse en `None` antes del `try`, porque si `open()` falla la variable nunca recibe un valor, y en el `finally` hay que verificarla antes de llamar a `close()`. Con `with` el cierre es automático y no hay que preocuparse por eso, por eso es la opción más clara y segura para la mayoría de los casos.

In [ ]:
# Versión con try-except-finally (cierre manual)
archivo = None
try:
    archivo = open("alumnos.txt", "rt", encoding="utf-8")
    for linea in archivo:
        print(linea.rstrip("\n"))
except FileNotFoundError:
    print("El archivo no existe.")
except OSError as error:
    print("No se pudo leer el archivo:", error)
finally:
    if archivo is not None:
        archivo.close()
        print("Archivo cerrado.")

# Versión equivalente con with (cierre automático)
try:
    with open("alumnos.txt", "rt", encoding="utf-8") as archivo:
        for linea in archivo:
            print(linea.rstrip("\n"))
except FileNotFoundError:
    print("El archivo no existe.")
except OSError as error:
    print("No se pudo leer el archivo:", error)

## 7. Desafío aplicado al Proyecto Integrador

Programa para gestionar `entregas.csv` con el formato `grupo;proyecto;responsable;avance`.

Funciones:
- a) `crear_entregas()`: crea el archivo con una única fila de encabezado.
- b) `cargar_entregas()`: carga entregas sin repetir el número de grupo (se ingresa 0 para terminar). Los grupos existentes se leen una sola vez al empezar la carga.
- c) `mostrar_entregas()`: muestra todas las entregas válidas.
- d) `buscar_entrega(grupo)`: busca por número de grupo y deja de leer apenas lo encuentra.
- e) `informar_bajo_avance()`: proyectos con avance menor al 50 %.
- f) `promedio_avance()`: calcula el promedio, devolviendo `None` si no hay registros válidos (para no dividir por cero).
- g) Todas las funciones de lectura informan y omiten los registros mal formados sin cortar el procesamiento.

Validaciones: grupo positivo, proyecto y responsable no vacíos, avance entre 0 y 100. Cada operación recorre el archivo una sola vez, línea por línea.

In [ ]:
ARCHIVO_ENTREGAS = "entregas.csv"
ENCABEZADO_ENTREGAS = "grupo;proyecto;responsable;avance"


# a) Crear el archivo con el encabezado
def crear_entregas():
    try:
        with open(ARCHIVO_ENTREGAS, "wt", encoding="utf-8") as archivo:
            archivo.write(ENCABEZADO_ENTREGAS + "\n")
        print("Archivo entregas.csv creado.")
        return True
    except OSError as error:
        print("No se pudo crear el archivo:", error)
        return False


# Validación de un registro (se usa en todas las lecturas)
def leer_entrega(linea):
    """Devuelve (grupo, proyecto, responsable, avance) o None si el registro es inválido."""
    campos = linea.rstrip("\n").split(";")
    if len(campos) != 4:
        return None
    try:
        grupo = int(campos[0])
        avance = float(campos[3])
    except ValueError:
        return None
    proyecto = campos[1].strip()
    responsable = campos[2].strip()
    if grupo <= 0 or proyecto == "" or responsable == "" or avance < 0 or avance > 100:
        return None
    return grupo, proyecto, responsable, avance


def informar_invalido(nro_linea, linea):
    print(f"Línea {nro_linea} mal formada, se omite: {linea.rstrip()}")


def mostrar_entrega(entrega):
    grupo, proyecto, responsable, avance = entrega
    print(f"Grupo {grupo:<4} | {proyecto:<25} | {responsable:<20} | {avance:.1f} %")


def grupos_registrados():
    """Devuelve un conjunto con los grupos ya cargados, o None si hubo un error."""
    grupos = set()
    try:
        with open(ARCHIVO_ENTREGAS, "rt", encoding="utf-8") as archivo:
            archivo.readline()  # se saltea el encabezado
            for linea in archivo:
                entrega = leer_entrega(linea)
                if entrega is not None:
                    grupos.add(entrega[0])
    except FileNotFoundError:
        # Si el archivo no existe, se crea con su encabezado
        if not crear_entregas():
            return None
    except OSError as error:
        print("No se pudo leer el archivo:", error)
        return None
    return grupos


# b) Cargar entregas sin repetir el número de grupo
def cargar_entregas():
    grupos = grupos_registrados()
    if grupos is None:
        return
    try:
        with open(ARCHIVO_ENTREGAS, "at", encoding="utf-8") as archivo:
            grupo = pedir_entero("Número de grupo (0 para terminar): ")
            while grupo != 0:
                if grupo < 0:
                    print("El número de grupo debe ser positivo.")
                elif grupo in grupos:
                    print(f"El grupo {grupo} ya tiene una entrega cargada.")
                else:
                    proyecto = input("Nombre del proyecto: ").strip()
                    responsable = input("Integrante responsable: ").strip()
                    avance = pedir_real("Porcentaje de avance (0 a 100): ")
                    if proyecto == "" or ";" in proyecto:
                        print("Nombre de proyecto inválido. No se grabó.")
                    elif responsable == "" or ";" in responsable:
                        print("Responsable inválido. No se grabó.")
                    elif avance < 0 or avance > 100:
                        print("El avance debe estar entre 0 y 100. No se grabó.")
                    else:
                        archivo.write(f"{grupo};{proyecto};{responsable};{avance}\n")
                        grupos.add(grupo)
                        print("Entrega grabada.")
                grupo = pedir_entero("Número de grupo (0 para terminar): ")
    except OSError as error:
        print("No se pudo escribir el archivo:", error)


# c) Mostrar todas las entregas válidas
def mostrar_entregas():
    try:
        with open(ARCHIVO_ENTREGAS, "rt", encoding="utf-8") as archivo:
            archivo.readline()
            nro_linea = 1
            cantidad = 0
            for linea in archivo:
                nro_linea += 1
                entrega = leer_entrega(linea)
                if entrega is None:
                    informar_invalido(nro_linea, linea)
                else:
                    mostrar_entrega(entrega)
                    cantidad += 1
            if cantidad == 0:
                print("No hay entregas válidas cargadas.")
    except FileNotFoundError:
        print("El archivo entregas.csv no existe.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)


# d) Buscar una entrega por número de grupo
def buscar_entrega(grupo_buscado):
    try:
        with open(ARCHIVO_ENTREGAS, "rt", encoding="utf-8") as archivo:
            archivo.readline()
            for linea in archivo:
                entrega = leer_entrega(linea)
                if entrega is not None and entrega[0] == grupo_buscado:
                    return entrega  # se deja de leer apenas se encuentra
    except FileNotFoundError:
        print("El archivo entregas.csv no existe.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)
    return None


# e) Proyectos con avance inferior al 50 %
def informar_bajo_avance(limite=50):
    try:
        with open(ARCHIVO_ENTREGAS, "rt", encoding="utf-8") as archivo:
            archivo.readline()
            nro_linea = 1
            encontrados = 0
            for linea in archivo:
                nro_linea += 1
                entrega = leer_entrega(linea)
                if entrega is None:
                    informar_invalido(nro_linea, linea)
                elif entrega[3] < limite:
                    mostrar_entrega(entrega)
                    encontrados += 1
            if encontrados == 0:
                print(f"No hay proyectos con avance menor al {limite} %.")
    except FileNotFoundError:
        print("El archivo entregas.csv no existe.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)


# f) Promedio de avance sin dividir por cero
def promedio_avance():
    suma = 0
    cantidad = 0
    try:
        with open(ARCHIVO_ENTREGAS, "rt", encoding="utf-8") as archivo:
            archivo.readline()
            for linea in archivo:
                entrega = leer_entrega(linea)
                if entrega is not None:
                    suma += entrega[3]
                    cantidad += 1
    except FileNotFoundError:
        print("El archivo entregas.csv no existe.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

    if cantidad == 0:
        return None
    return suma / cantidad

In [ ]:
def menu_entregas():
    opcion = -1
    while opcion != 0:
        print("\n=== GESTIÓN DE ENTREGAS ===")
        print("1. Crear archivo (borra lo existente)")
        print("2. Cargar entregas")
        print("3. Mostrar entregas")
        print("4. Buscar por número de grupo")
        print("5. Proyectos con avance menor al 50 %")
        print("6. Promedio de avance")
        print("0. Salir")
        opcion = pedir_entero("Opción: ")

        if opcion == 1:
            crear_entregas()
        elif opcion == 2:
            cargar_entregas()
        elif opcion == 3:
            mostrar_entregas()
        elif opcion == 4:
            grupo = pedir_entero("Grupo a buscar: ")
            entrega = buscar_entrega(grupo)
            if entrega is None:
                print(f"No se encontró una entrega del grupo {grupo}.")
            else:
                mostrar_entrega(entrega)
        elif opcion == 5:
            informar_bajo_avance()
        elif opcion == 6:
            promedio = promedio_avance()
            if promedio is None:
                print("No hay registros válidos para calcular el promedio.")
            else:
                print(f"Promedio de avance: {promedio:.2f} %")
        elif opcion != 0:
            print("Opción inválida.")
    print("Fin del programa.")


menu_entregas()